<a href="https://colab.research.google.com/github/rskhoshnaw/Lumina/blob/main/Lumina_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# پچ کردن خودکار کامپوننت‌های کارت در Lumina
cards_file = "/content/Lumina_v2/lumina/cards.py"

with open(cards_file, "r", encoding="utf-8") as f:
    cards_code = f.read()

# ۱. جایگزینی تمام ارجاعات DejaVu Sans با Noto Naskh Arabic
cards_code = cards_code.replace('"DejaVu Sans"', '"Noto Naskh Arabic"')
cards_code = cards_code.replace("'DejaVu Sans'", "'Noto Naskh Arabic'")

# ۲. جلوگیری از ارسال فونت‌های ناسازگار به متد QuoteCard
cards_code = cards_code.replace(
    "def __init__(self, text_content, author,",
    "def __init__(self, text_content, author, font_name='Noto Naskh Arabic',"
)

# ۳. اجبار استفاده از Noto در ابتدای تابع QuoteCard
patch_injection = """
        # قفل کردن روی فونت سازگار کُردی
        font_name = "Noto Naskh Arabic"
"""
if "font_name = \"Noto Naskh Arabic\"" not in cards_code:
    cards_code = cards_code.replace(
        "self.font_name = font_name",
        f"font_name = 'Noto Naskh Arabic'\n        self.font_name = font_name"
    )

with open(cards_file, "w", encoding="utf-8") as f:
    f.write(cards_code)

print("✅ فایل lumina/cards.py با موفقیت به‌روزرسانی و سازگار شد!")

✅ فایل lumina/cards.py با موفقیت به‌روزرسانی و سازگار شد!


# 1. نصب پیش‌نیازهای گرافیکی، لاتک و فونت Noto Naskh


In [ ]:
# 1. نصب پیش‌نیازهای گرافیکی، لاتک و فونت Noto Naskh
!sudo apt update
!sudo apt install -y libcairo2-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra fonts-noto-arabic

In [7]:
# پچ کردن خودکار کامپوننت‌های کارت در Lumina
cards_file = "/content/Lumina_v2/lumina/cards.py"

with open(cards_file, "r", encoding="utf-8") as f:
    cards_code = f.read()

# ۱. جایگزینی تمام ارجاعات DejaVu Sans با Noto Naskh Arabic
cards_code = cards_code.replace('"DejaVu Sans"', '"Noto Naskh Arabic"')
cards_code = cards_code.replace("'DejaVu Sans'", "'Noto Naskh Arabic'")

# ۲. جلوگیری از ارسال فونت‌های ناسازگار به متد QuoteCard
cards_code = cards_code.replace(
    "def __init__(self, text_content, author,",
    "def __init__(self, text_content, author, font_name='Noto Naskh Arabic',"
)

# ۳. اجبار استفاده از Noto در ابتدای تابع QuoteCard
patch_injection = """
        # قفل کردن روی فونت سازگار کُردی
        font_name = "Noto Naskh Arabic"
"""
if "font_name = \"Noto Naskh Arabic\"" not in cards_code:
    cards_code = cards_code.replace(
        "self.font_name = font_name",
        f"font_name = 'Noto Naskh Arabic'\n        self.font_name = font_name"
    )

with open(cards_file, "w", encoding="utf-8") as f:
    f.write(cards_code)

print("✅ فایل lumina/cards.py با موفقیت به‌روزرسانی و سازگار شد!")

✅ فایل lumina/cards.py با موفقیت به‌روزرسانی و سازگار شد!


In [8]:
# خط زیر را به بخش MANDATORY RULES در پرامپت app.py اضافه کنید:
"4. NEVER use font_name='DejaVu Sans'. If using QuoteCard or any card, always set font_name='Noto Naskh Arabic'."

"4. NEVER use font_name='DejaVu Sans'. If using QuoteCard or any card, always set font_name='Noto Naskh Arabic'."

In [10]:
%%writefile /content/Lumina_v2/lumina/cards.py
"""RTL-optimized Cards and Visual Components for Lumina Studio."""
from manim import *
from arabic_reshaper import reshape
from bidi.algorithm import get_display

def rtl_text(text, font_size=32, color=WHITE, font="Noto Naskh Arabic", **kwargs):
    """پردازش ایمن متن راست‌به‌چپ با انیمیشن معکوس."""
    if not text or not str(text).strip():
        text = " "
    reshaped = reshape(str(text))
    bidi_text = get_display(reshaped)
    t = Text(bidi_text, font=font, font_size=font_size, color=color, **kwargs)
    if len(t.submobjects) > 0:
        t.submobjects.reverse()
    return t


class TitleCard(VGroup):
    """کارت عنوان اصلی ویدیو با هدر و تراز راست‌چین مدرن."""
    def __init__(self, title, subtitle=None, category=None, width=10.5, height=5.5, **kwargs):
        super().__init__(**kwargs)

        # ۱. پس‌زمینه کارت با حاشیه نرم
        self.background = RoundedRectangle(
            width=width,
            height=height,
            corner_radius=0.3,
            fill_color="#141419",
            fill_opacity=0.9,
            stroke_color="#2A2A38",
            stroke_width=2
        )
        self.add(self.background)

        # ۲. نوار نشانگر رنگی در سمت راست کارت (سمت شروع متن در RTL)
        accent_bar = RoundedRectangle(
            width=0.15,
            height=height - 0.8,
            corner_radius=0.07,
            fill_color=BLUE_C,
            fill_opacity=1,
            stroke_width=0
        )
        accent_bar.align_to(self.background, RIGHT).shift(LEFT * 0.4)
        self.add(accent_bar)

        # ۳. برچسب دسته‌بندی (اختیاری)
        content_elements = []
        if category:
            cat_badge = rtl_text(f"• {category}", font_size=20, color=BLUE_B)
            content_elements.append(cat_badge)

        # ۴. عنوان اصلی
        title_obj = rtl_text(title, font_size=42, color=WHITE)
        content_elements.append(title_obj)

        # ۵. زیرعنوان (اختیاری)
        if subtitle:
            sub_obj = rtl_text(subtitle, font_size=26, color=GRAY_B)
            content_elements.append(sub_obj)

        # چیدمان عمودی و راست‌چین المان‌ها نسبت به نوار راست
        self.text_group = VGroup(*content_elements).arrange(DOWN, aligned_edge=RIGHT, buff=0.4)
        self.text_group.next_to(accent_bar, LEFT, buff=0.45)
        self.add(self.text_group)


class QuoteCard(VGroup):
    """کارت نقل‌قول یا جمع‌بندی با خط کناری و جایگاه متناسب نویسنده."""
    def __init__(self, text_content, author=None, font_name="Noto Naskh Arabic", width=10.0, height=4.2, **kwargs):
        super().__init__(**kwargs)

        # ۱. پس‌زمینه کارت
        self.background = RoundedRectangle(
            width=width,
            height=height,
            corner_radius=0.25,
            fill_color="#121316",
            fill_opacity=0.92,
            stroke_color=YELLOW_D,
            stroke_width=1.5
        )
        self.add(self.background)

        # ۲. علامت نقل‌قول در گوشه بالا-راست
        quote_mark = rtl_text("”", font_size=64, color=YELLOW_D).set_opacity(0.4)
        quote_mark.align_to(self.background, UR).shift(DOWN * 0.25 + LEFT * 0.5)
        self.add(quote_mark)

        # ۳. متن نقل‌قول
        self.quote_text = rtl_text(text_content, font_size=28, color="#EEEEEE")
        # محدودسازی عرض متن برای جلوگیری از بیرون‌زدگی از کادر
        if self.quote_text.width > (width - 1.8):
            self.quote_text.scale_to_fit_width(width - 1.8)

        self.quote_text.align_to(self.background, RIGHT).shift(LEFT * 0.9)
        self.quote_text.align_to(self.background, UP).shift(DOWN * 1.0)
        self.add(self.quote_text)

        # ۴. نویسنده / منبع نقل‌قول در پایین-چپ (یا پایین-راست با تراز منظم)
        if author:
            self.author_text = rtl_text(f"— {author}", font_size=24, color=YELLOW)
            self.author_text.next_to(self.quote_text, DOWN, buff=0.45, aligned_edge=RIGHT)
            self.add(self.author_text)

print("✅ lumina/cards.py بە شێوازی راست‌چین و نوێ بەسەرکەوتوویی نوێکرایەوە!")

Overwriting /content/Lumina_v2/lumina/cards.py


# 2. کلون مخزن و نصب پکیج‌های سازگار


In [2]:
# 2. کلون مخزن و نصب پکیج‌های سازگار
import os
if not os.path.exists('Lumina_v2'):
    !git clone https://github.com/rskhoshnaw/Lumina_v2.git
os.chdir('Lumina_v2')
!pip install --upgrade pip
!pip install manim==0.21.0 google-genai gradio==5.45.0 arabic-reshaper==3.0.0 python-bidi==0.4.2 websockets==14.2 pillow==11.1.0 huggingface_hub>=0.26.0

# 3. مانت کردن گوگل درایو

In [3]:
# 3. مانت کردن گوگل درایو
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/LuminaStudio/videos', exist_ok=True)
print('✅ Google Drive Mounted Successfully.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive Mounted Successfully.


# 4. بارگذاری امن کلید Gemini API

In [4]:
# 4. بارگذاری امن کلید Gemini API
from google.colab import userdata
import os
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('✅ API Key loaded successfully from Colab Secrets.')
except Exception as e:
    print('❌ Error: لطفاً GEMINI_API_KEY را در بخش Secrets سایدبار کولب وارد کنید.')

✅ API Key loaded successfully from Colab Secrets.


# سلول ۵: ساخت فایل یکپارچه app.py (شامل پچ و پورت استاندارد)

In [5]:
%%writefile app.py
"""Gradio UI for Lumina Studio 2.0 - Native Kurdish Support."""
import shutil
from pathlib import Path
import gradio as gr
from core.gemini_provider import GeminiProvider
from core.pipeline import LuminaPipeline

def patched_generate_script(self, prompt, category="Physics", language="ckb", audience="General", style="Modern", duration_seconds=20):
    full_prompt = f"""
You are an expert Manim animator. Generate valid Python code for Lumina Studio v2.0.

Topic: {prompt}
Category: {category}
Language: {language} (Kurdish Sorani)
Target Duration: ~{duration_seconds} seconds
Style: {style}

CRITICAL RULES FOR KURDISH / RTL:
1. DO NOT import or use `arabic_reshaper` or `bidi.algorithm`. Pango handles Kurdish natively.
2. For ANY standalone text, define and use this exact helper:
   def rtl_text(text, font_size=32, color=WHITE, font="Noto Naskh Arabic", **kwargs):
       t = Text(str(text), font=font, font_size=font_size, color=color, **kwargs)
       if len(t.submobjects) > 1:
           t.submobjects.sort(key=lambda m: -m.get_center()[0])
       return t

3. CARD RULES:
   - FormulaHighlightCard(latex_str=r"...", label_text="دەقی کوردی", font_name="Noto Naskh Arabic")
   - QuoteCard(text_content="دەقی وتەکە", author="ناوی نووسەر", font_name="Noto Naskh Arabic")
   - NEVER use parameters like 'formula' or 'title' in FormulaHighlightCard.

4. IMPORTS:
   from manim import *
   from lumina import LuminaScene, QuoteCard, FormulaHighlightCard, TitleCard

5. Return ONLY executable Python code inside ```python ... ``` blocks.
"""
    return self.generate_text(full_prompt)

GeminiProvider.generate_script = patched_generate_script
pipeline = LuminaPipeline()

def generate_video_ui(prompt, category, language, audience, style, duration):
    try:
        result = pipeline.run(
            prompt=prompt,
            category=category,
            language=language,
            audience=audience,
            style=style,
            duration_seconds=int(duration),
            quality="qm"
        )
        video_path = result.get("video_file")
        if video_path and Path(video_path).exists():
            drive_dest = Path("/content/drive/MyDrive/LuminaStudio/videos") / Path(video_path).name
            if drive_dest.parent.exists():
                shutil.copy(video_path, drive_dest)
            return video_path, "✅ ڤیدیۆکە بە سەرکەوتوویی دروستکرا!"
        return None, "❌ هەڵە: فایلی ڤیدیۆ نەدۆزرایەوە."
    except Exception as e:
        return None, f"❌ هەڵەی سیستم:\n{str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 Lumina Studio 2.0")
    gr.Markdown("پلاتفۆرمی دروستکردنی ڤیدیۆی فێرکاری بە Manim و Gemini بە زمانی کوردی.")

    with gr.Row():
        with gr.Column(scale=1):
            prompt_input = gr.Textbox(
                label="بابەت یان سیناریۆی ڤیدیۆ",
                placeholder="بۆ نموونە: یاسای کێشکردنی گەردوونی نیوتن بە کوردی...",
                lines=3
            )
            category_input = gr.Dropdown(
                choices=["Physics", "Mathematics", "Computer Science", "General"],
                value="Physics",
                label="پۆلێن (Category)"
            )
            language_input = gr.Dropdown(
                choices=["ckb", "fa", "en"],
                value="ckb",
                label="زمان (Language)"
            )
            audience_input = gr.Textbox(
                value="قوتابیانی ئامادەیی",
                label="ئاستی وەرگر (Audience)"
            )
            style_input = gr.Dropdown(
                choices=["Cinematic & Modern", "Minimalist", "Academic"],
                value="Cinematic & Modern",
                label="شێوازی بینراو (Style)"
            )
            duration_slider = gr.Slider(
                minimum=10, maximum=60, step=5, value=20,
                label="ماوەی خەمڵێنراو (چرکە)"
            )
            submit_btn = gr.Button("🚀 دروستکردنی ڤیدیۆ", variant="primary")

        with gr.Column(scale=1):
            status_box = gr.Textbox(label="دۆخی پرۆسەسەکان", lines=4)
            video_output = gr.Video(label="پێشبینینی ڤیدیۆ")

    submit_btn.click(
        fn=generate_video_ui,
        inputs=[prompt_input, category_input, language_input, audience_input, style_input, duration_slider],
        outputs=[video_output, status_box]
    )

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860, share=True)

Overwriting app.py


#سلول ۶: اجرای نهایی سرور Gradio

In [24]:
# 6. اجرای پایپلاین و دریافت لینک عمومی
!python app.py

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://6f69eb230ac2a90ff6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
[Repair Loop] Error detected. Attempting repair 1/2...
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 3022, in block_thread
    time.sleep(0.1)
    ~~~~~~~~~~^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/Lumina_v2/app.py", line 111, in <module>
    demo.launch(server_name="0.0.0.0", server_port=7860, share=True)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2919, in la